In [1]:
import os
from pathlib import Path

# Forcefully hunt down Java 17 to prevent JVM crashes
possible_paths = [
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/opt/homebrew/opt/openjdk@17",
    "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home",
    "/Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home"
]

java_home = None
for path in possible_paths:
    if Path(path).exists() and Path(path + "/bin/java").exists():
        java_home = path
        break

if java_home:
    os.environ["JAVA_HOME"] = java_home
    print(f"Environment Locked. JAVA_HOME: {os.environ['JAVA_HOME']}")
else:
    print("Warning: Java 17 not found in standard paths.")

Environment Locked. JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


In [2]:
from pyspark.sql import SparkSession

# Boot up the Spark Engine (using 4g memory to ensure stability)
spark = SparkSession.builder \
    .appName("bixi-spatial-enrichment") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "96") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "America/Toronto")
print(f"Spark Version: {spark.version} initialized successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 10:15:24 WARN Utils: Your hostname, Comanes-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.249.35.225 instead (on interface en0)
26/03/28 10:15:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 10:15:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/28 10:15:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version: 4.1.1 initialized successfully.


In [3]:
!pip install requests

In [7]:
%pip install pyarrow 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 24.6 MB/s  0:00:01 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [19]:
!pip install openmeteo_requests

  Using cached openmeteo_requests-1.7.5-py3-none-any.whl.metadata (11 kB)
  Using cached niquests-3.18.2-py3-none-any.whl.metadata (15 kB)
  Using cached openmeteo_sdk-1.26.0-py3-none-any.whl.metadata (935 bytes)
  Using cached urllib3_future-2.18.901-py3-none-any.whl.metadata (16 kB)
  Using cached wassima-2.0.5-py3-none-any.whl.metadata (3.7 kB)
  Using cached jh2-5.0.10-cp37-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (4.0 kB)
  Using cached qh3-1.7.0-cp37-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (5.0 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
Using cached openmeteo_requests-1.7.5-py3-none-any.whl (7.1 kB)
Using cached niquests-3.18.2-py3-none-any.whl (207 kB)
Using cached urllib3_future-2.18.901-py3-none-any.whl (713 kB)
Using cached jh2-5.0.10-cp37-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl (622 kB)
Using cached qh3-1.7.0-cp37-abi3-macosx_10_

In [23]:
!pip install requests_cache
!pip install retry
!pip install retry_requests

In [27]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
from pathlib import Path

# Setup the API client
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Montreal Coordinates
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 45.5088,
    "longitude": -73.5617,
    "start_date": "2024-01-01",
    "end_date": "2025-12-31",
    "hourly": ["temperature_2m", "precipitation"],
    "timezone": "America/New_York"
}

print("Fetching weather data...")
responses = openmeteo.weather_api(url, params=params)
response = responses[0]

# Process hourly data
hourly = response.Hourly()
hourly_data = {
    "ts_hour": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    ),
    "temp": hourly.Variables(0).ValuesAsNumpy(), # Renamed to 'temp' for the join
    "precip": hourly.Variables(1).ValuesAsNumpy()
}

weather_df = pd.DataFrame(data = hourly_data)

# 1. Localize and remove offset
weather_df['ts_hour'] = weather_df['ts_hour'].dt.tz_convert('America/New_York').dt.tz_localize(None)

# 2. THE CRITICAL FIX: Cast to microseconds (Spark-compatible)
weather_df['ts_hour'] = weather_df['ts_hour'].astype('datetime64[us]')

# 3. Save to the file path
output_dir = Path("/Users/comanetan/projects/bixi-analytics/data/silver/weather_hourly")
output_dir.mkdir(parents=True, exist_ok=True)
file_path = output_dir / "weather.parquet"

weather_df.to_parquet(file_path, index=False)
print(f"✅ Corrected weather data saved to: {file_path}")

Fetching weather data...
✅ Corrected weather data saved to: /Users/comanetan/projects/bixi-analytics/data/silver/weather_hourly/weather.parquet


In [30]:
import os
from pathlib import Path
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# ---------------------------------------------------------------------
# 1. Path Configuration
# ---------------------------------------------------------------------
base_path = Path("/Users/comanetan/projects/bixi-analytics")
rides_path = f"file://{str(base_path / 'data' / 'silver' / 'rides')}"
community_path = f"file://{str(base_path / 'data' / 'feature_temp' / 'community_assignments')}"
mapping_path = str(base_path / "data" / "silver" / "station_cleaning" / "station_direct_match_mapping")
weather_path = f"file://{str(base_path / 'data' / 'silver' / 'weather_hourly')}"

# ---------------------------------------------------------------------
# 2. Load and Aggregate Rides
# ---------------------------------------------------------------------
print("Reading historical rides...")
df_rides = spark.read.parquet(rides_path)

# Use start_station_key to aggregate (based on your earlier schema error)
df_demand = df_rides.withColumn("ts_hour", F.date_trunc("hour", F.col("start_time_ms"))) \
    .groupBy("ts_hour", "start_station_key") \
    .agg(F.count("*").alias("demand_count")) \
    .withColumnRenamed("start_station_key", "station_id")

# ---------------------------------------------------------------------
# 3. Add Holiday & Calendar Features
# ---------------------------------------------------------------------
print("📅 Adding holiday and calendar features...")
holidays = ["2024-06-24", "2024-07-01", "2024-09-02", "2024-10-14", 
            "2025-06-24", "2025-07-01", "2025-09-02", "2025-10-13"]

df_enriched = df_demand \
    .withColumn("day_of_week", F.dayofweek("ts_hour")) \
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("is_holiday", F.when(F.to_date("ts_hour").cast("string").isin(holidays), 1).otherwise(0))

# ---------------------------------------------------------------------
# 4. Join Communities and Weather
# ---------------------------------------------------------------------
print("🔗 Joining Communities and Weather...")

df_comm = spark.read.parquet(community_path) \
    .withColumnRenamed("id", "station_id") \
    .withColumnRenamed("label", "community_id")

df_enriched = df_enriched.join(F.broadcast(df_comm), on="station_id", how="left")

try:
    weather_df = spark.read.parquet(weather_path)
    # Ensure weather_df has 'temp' and 'precip' columns
    df_enriched = df_enriched.join(F.broadcast(weather_df), on="ts_hour", how="left")
    print("✅ Weather join successful.")
except Exception as e:
    print(f"⚠️ Weather join failed: {e}")

# ---------------------------------------------------------------------
# 5. Smart Coordinate Mapping (Fuzzy Column Logic)
# ---------------------------------------------------------------------
print("📍 Resolving station coordinates...")
mapping_pd = pd.read_parquet(mapping_path)

# Fuzzy Search for Columns
# We look for the column that contains "STN_" or actual IDs, not coordinates
def find_col(possible_names, df_cols):
    for p in possible_names:
        for c in df_cols:
            if p.lower() in c.lower(): return c
    return None

id_col = find_col(['station_key', 'station_id', 'canonical'], mapping_pd.columns)
lat_col = find_col(['lat', 'latitude'], mapping_pd.columns)
lon_col = find_col(['lon', 'longitude'], mapping_pd.columns)

if id_col and lat_col and lon_col:
    # Build clean mapping and force station_id to string to match df_enriched
    clean_map = mapping_pd[[id_col, lat_col, lon_col]].copy()
    clean_map.columns = ['station_id', 'lat', 'lon']
    clean_map['station_id'] = clean_map['station_id'].astype(str)
    
    # Convert to Spark
    spark_mapping = spark.createDataFrame(clean_map)
    # Join with the aggregated data
    df_final = df_enriched.join(F.broadcast(spark_mapping), on="station_id", how="left")
else:
    print(f"⚠️ Mapping failed. Found cols: {mapping_pd.columns.tolist()}")
    df_final = df_enriched

# ---------------------------------------------------------------------
# 6. Final Save
# ---------------------------------------------------------------------
output_dir = base_path / "data" / "feature_temp" / "model_features_final"
df_final.write.mode("overwrite").parquet(f"file://{str(output_dir)}")

print(f"🚀 DONE! Final dataset saved.")
df_final.select("ts_hour", "station_id", "demand_count", "temp", "precip").show(5)

Reading historical rides...
📅 Adding holiday and calendar features...
🔗 Joining Communities and Weather...
✅ Weather join successful.
📍 Resolving station coordinates...


🚀 DONE! Final dataset saved.


+-------------------+--------------------+------------+-----+------+
|            ts_hour|          station_id|demand_count| temp|precip|
+-------------------+--------------------+------------+-----+------+
|2024-06-24 16:00:00|45.524790,-73.565...|           9| 21.4|   0.0|
|2024-10-19 18:00:00|45.506539,-73.576...|          18| 13.9|   0.0|
|2024-10-19 15:00:00|45.470400,-73.565...|           6|16.25|   0.0|
|2024-10-19 16:00:00|45.527831,-73.571...|          15| 16.4|   0.0|
|2024-10-19 15:00:00|45.483249,-73.591...|           5|16.25|   0.0|
+-------------------+--------------------+------------+-----+------+
only showing top 5 rows


In [31]:
# 1. Check the full list of columns and their types
print("--- Final Dataset Schema ---")
df_final.printSchema()

# 2. View a wider sample of the actual data
print("--- Data Preview (Top 5 Rows) ---")
# Using vertical=True makes it much easier to read when you have many columns
df_final.show(5, truncate=False, vertical=True)

# 3. Quick count check to ensure weather isn't all NULLs
null_weather_count = df_final.filter(F.col("temp").isNull()).count()
total_count = df_final.count()

print(f"Total Rows: {total_count}")
print(f"Rows missing weather data: {null_weather_count}")

--- Final Dataset Schema ---
root
 |-- station_id: string (nullable = true)
 |-- ts_hour: timestamp (nullable = true)
 |-- demand_count: long (nullable = false)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- is_holiday: integer (nullable = false)
 |-- community_id: long (nullable = true)
 |-- temp: float (nullable = true)
 |-- precip: float (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)

--- Data Preview (Top 5 Rows) ---


-RECORD 0-------------------------------------------------------------------
 station_id   | 45.524790,-73.565450|calixa-lavallée / sherbrooke           
 ts_hour      | 2024-06-24 16:00:00                                         
 demand_count | 9                                                           
 day_of_week  | 2                                                           
 is_weekend   | 0                                                           
 is_holiday   | 1                                                           
 community_id | NULL                                                        
 temp         | 21.4                                                        
 precip       | 0.0                                                         
 lat          | 45.52479                                                    
 lon          | -73.56545                                                   
-RECORD 1-------------------------------------------------------------------

Total Rows: 6179615
Rows missing weather data: 38329


In [32]:
import pyspark.sql.functions as F

# 1. Full Schema Check
print("--- 📋 Final Feature Schema ---")
df_final.printSchema()

# 2. Detailed Data Inspection (Vertical view is better for many columns)
print("--- 🔍 Vertical Data Preview (First 2 Rows) ---")
df_final.show(2, truncate=False, vertical=True)

# 3. Data Quality Report
print("--- 📊 Data Quality Summary ---")
total_rows = df_final.count()

# Calculate fill rates for key features
quality_stats = df_final.select(
    (F.count("community_id") / total_rows * 100).alias("community_fill_rate"),
    (F.count("temp") / total_rows * 100).alias("weather_fill_rate"),
    (F.count("lat") / total_rows * 100).alias("coords_fill_rate"),
    F.sum("is_holiday").alias("total_holiday_hours")
).collect()[0]

print(f"✅ Total Records: {total_rows:,}")
print(f"🏘️ Community Mapping: {quality_stats['community_fill_rate']:.2f}%")
print(f"🌡️ Weather Coverage: {quality_stats['weather_fill_rate']:.2f}%")
print(f"📍 Coordinate Match: {quality_stats['coords_fill_rate']:.2f}%")
print(f"🎉 Holiday Flags: {quality_stats['total_holiday_hours']} hourly records")

# 4. Final Verification of the Holiday Logic (e.g., St-Jean-Baptiste)
print("\n--- 🇨🇦 Holiday Logic Validation (June 24, 2024) ---")
df_final.filter("to_date(ts_hour) = '2024-06-24'") \
        .select("ts_hour", "is_holiday") \
        .distinct() \
        .show(1)

--- 📋 Final Feature Schema ---
root
 |-- station_id: string (nullable = true)
 |-- ts_hour: timestamp (nullable = true)
 |-- demand_count: long (nullable = false)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- is_holiday: integer (nullable = false)
 |-- community_id: long (nullable = true)
 |-- temp: float (nullable = true)
 |-- precip: float (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)

--- 🔍 Vertical Data Preview (First 2 Rows) ---


-RECORD 0---------------------------------------------------------
 station_id   | 45.524790,-73.565450|calixa-lavallée / sherbrooke 
 ts_hour      | 2024-06-24 16:00:00                               
 demand_count | 9                                                 
 day_of_week  | 2                                                 
 is_weekend   | 0                                                 
 is_holiday   | 1                                                 
 community_id | NULL                                              
 temp         | 21.4                                              
 precip       | 0.0                                               
 lat          | 45.52479                                          
 lon          | -73.56545                                         
-RECORD 1---------------------------------------------------------
 station_id   | 45.506539,-73.576609|university / milton          
 ts_hour      | 2024-10-19 18:00:00                           

✅ Total Records: 6,179,615
🏘️ Community Mapping: 0.00%
🌡️ Weather Coverage: 99.38%
📍 Coordinate Match: 99.87%
🎉 Holiday Flags: 97313 hourly records

--- 🇨🇦 Holiday Logic Validation (June 24, 2024) ---
+-------------------+----------+
|            ts_hour|is_holiday|
+-------------------+----------+
|2024-06-24 10:00:00|         1|
+-------------------+----------+
only showing top 1 row


In [33]:
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
import pyspark.sql.functions as F

# Complete Montreal Metro (STM) Station List
metro_data = [
    ("Angrignon", 45.4463, -73.6038), ("Monk", 45.4511, -73.5932), ("Jolicoeur", 45.4568, -73.5821),
    ("Verdun", 45.4591, -73.5714), ("De l'Église", 45.4628, -73.5670), ("LaSalle", 45.4712, -73.5658),
    ("Charlevoix", 45.4785, -73.5697), ("Lionel-Groulx", 45.4829, -73.5825), ("Atwater", 45.4891, -73.5845),
    ("Guy-Concordia", 45.4951, -73.5797), ("Peel", 45.5009, -73.5747), ("McGill", 45.5041, -73.5715),
    ("Place-des-Arts", 45.5082, -73.5684), ("Saint-Laurent", 45.5111, -73.5649), ("Berri-UQAM", 45.5155, -73.5607),
    ("Beaudry", 45.5191, -73.5570), ("Papineau", 45.5239, -73.5532), ("Frontenac", 45.5330, -73.5521),
    ("Préfontaine", 45.5416, -73.5540), ("Joliette", 45.5469, -73.5516), ("Pie-IX", 45.5539, -73.5517),
    ("Viau", 45.5611, -73.5468), ("Assomption", 45.5694, -73.5469), ("Cadillac", 45.5768, -73.5463),
    ("Langelier", 45.5826, -73.5432), ("Radisson", 45.5888, -73.5358), ("Honoré-Beaugrand", 45.5967, -73.5352),
    ("Henri-Bourassa", 45.5543, -73.6675), ("Sauvé", 45.5507, -73.6560), ("Crémazie", 45.5458, -73.6472),
    ("Jarry", 45.5434, -73.6288), ("Jean-Talon", 45.5398, -73.6134), ("Beaubien", 45.5350, -73.6047),
    ("Rosemont", 45.5312, -73.5979), ("Laurier", 45.5281, -73.5888), ("Mont-Royal", 45.5247, -73.5822),
    ("Sherbrooke", 45.5186, -73.5684), ("Champ-de-Mars", 45.5102, -73.5562), ("Place-d'Armes", 45.5056, -73.5587),
    ("Square-Victoria-OACI", 45.5021, -73.5623), ("Bonaventure", 45.4982, -73.5674), ("Lucien-L'Allier", 45.4950, -73.5714),
    ("Georges-Vanier", 45.4889, -73.5767), ("Place-Saint-Henri", 45.4772, -73.5861), ("Vendôme", 45.4741, -73.6037),
    ("Villa-Maria", 45.4795, -73.6190), ("Snowdon", 45.4855, -73.6277), ("Côte-Sainte-Catherine", 45.4925, -73.6327),
    ("Plamondon", 45.4952, -73.6418), ("Namur", 45.4953, -73.6534), ("De la Savane", 45.5005, -73.6625),
    ("Du Collège", 45.5094, -73.6749), ("Côte-Vertu", 45.5143, -73.6821), ("Longueuil-U-de-S", 45.5245, -73.5215),
    ("Jean-Drapeau", 45.5134, -73.5332), ("Saint-Michel", 45.5597, -73.5997), ("D'Iberville", 45.5535, -73.6022),
    ("Fabre", 45.5469, -73.6074), ("De Castelnau", 45.5355, -73.6198), ("Parc", 45.5302, -73.6247),
    ("Acadie", 45.5235, -73.6234), ("Outremont", 45.5201, -73.6149), ("Édouard-Montpetit", 45.5100, -73.6128),
    ("Université-de-Montréal", 45.5030, -73.6181), ("Côte-des-Neiges", 45.4970, -73.6234), ("Cartier", 45.5604, -73.6817),
    ("De la Concorde", 45.5609, -73.7103), ("Montmorency", 45.5583, -73.7212)
]

schema = StructType([
    StructField("metro_name", StringType(), True),
    StructField("m_lat", DoubleType(), True),
    StructField("m_lon", DoubleType(), True)
])

df_metro = spark.createDataFrame(metro_data, schema)

In [34]:
from pyspark.sql.window import Window

# 1. Cross Join BIXI stations with Metro stations (Broadcast for speed)
# We only need distinct station IDs to save computation
df_stations = df_final.select("station_id", "lat", "lon").distinct()
df_cross = df_stations.crossJoin(F.broadcast(df_metro))

# 2. Haversine Calculation (Result in Kilometers)
def haversine_dist(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth's radius in km
    dlat = F.radians(lat2 - lat1)
    dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat / 2)**2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2)**2
    return 2 * R * F.atan2(F.sqrt(a), F.sqrt(1 - a))

df_dist = df_cross.withColumn("dist_km", haversine_dist(F.col("lat"), F.col("lon"), F.col("m_lat"), F.col("m_lon")))

# 3. Find the nearest Metro for every BIXI station
window_spec = Window.partitionBy("station_id").orderBy("dist_km")
df_nearest = df_dist.withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select("station_id", "metro_name", "dist_km")

# 4. Create the final proximity clusters
df_final = df_final.join(df_nearest, on="station_id", how="left") \
    .withColumn("metro_proximity", 
        F.when(F.col("dist_km") <= 0.3, "Very High")
         .when(F.col("dist_km") <= 0.8, "High")
         .otherwise("Low")
    )

In [35]:
# Check how many stations fall into each cluster
df_final.select("station_id", "metro_proximity").distinct() \
    .groupBy("metro_proximity") \
    .count().show()

# Sample check: Average demand near vs far from Metro
df_final.groupBy("metro_proximity") \
    .agg(F.avg("demand_count").alias("avg_hourly_demand")) \
    .orderBy("avg_hourly_demand", ascending=False).show()

+---------------+-----+
|metro_proximity|count|
+---------------+-----+
|            Low|  818|
|           High|  623|
|      Very High|  443|
+---------------+-----+



+---------------+-----------------+
|metro_proximity|avg_hourly_demand|
+---------------+-----------------+
|      Very High|5.142792316582324|
|           High|4.651205714747891|
|            Low|3.679807721031197|
+---------------+-----------------+



In [38]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# 1. Define the Full Metro Dataset (68 Stations)
metro_data = [
    ("Angrignon", 45.4463, -73.6038), ("Monk", 45.4511, -73.5932), ("Jolicoeur", 45.4568, -73.5821),
    ("Verdun", 45.4591, -73.5714), ("De l'Église", 45.4628, -73.5670), ("LaSalle", 45.4712, -73.5658),
    ("Charlevoix", 45.4785, -73.5697), ("Lionel-Groulx", 45.4829, -73.5825), ("Atwater", 45.4891, -73.5845),
    ("Guy-Concordia", 45.4951, -73.5797), ("Peel", 45.5009, -73.5747), ("McGill", 45.5041, -73.5715),
    ("Place-des-Arts", 45.5082, -73.5684), ("Saint-Laurent", 45.5111, -73.5649), ("Berri-UQAM", 45.5155, -73.5607),
    ("Beaudry", 45.5191, -73.5570), ("Papineau", 45.5239, -73.5532), ("Frontenac", 45.5330, -73.5521),
    ("Préfontaine", 45.5416, -73.5540), ("Joliette", 45.5469, -73.5516), ("Pie-IX", 45.5539, -73.5517),
    ("Viau", 45.5611, -73.5468), ("Assomption", 45.5694, -73.5469), ("Cadillac", 45.5768, -73.5463),
    ("Langelier", 45.5826, -73.5432), ("Radisson", 45.5888, -73.5358), ("Honoré-Beaugrand", 45.5967, -73.5352),
    ("Henri-Bourassa", 45.5543, -73.6675), ("Sauvé", 45.5507, -73.6560), ("Crémazie", 45.5458, -73.6472),
    ("Jarry", 45.5434, -73.6288), ("Jean-Talon", 45.5398, -73.6134), ("Beaubien", 45.5350, -73.6047),
    ("Rosemont", 45.5312, -73.5979), ("Laurier", 45.5281, -73.5888), ("Mont-Royal", 45.5247, -73.5822),
    ("Sherbrooke", 45.5186, -73.5684), ("Champ-de-Mars", 45.5102, -73.5562), ("Place-d'Armes", 45.5056, -73.5587),
    ("Square-Victoria-OACI", 45.5021, -73.5623), ("Bonaventure", 45.4982, -73.5674), ("Lucien-L'Allier", 45.4950, -73.5714),
    ("Georges-Vanier", 45.4889, -73.5767), ("Place-Saint-Henri", 45.4772, -73.5861), ("Vendôme", 45.4741, -73.6037),
    ("Villa-Maria", 45.4795, -73.6190), ("Snowdon", 45.4855, -73.6277), ("Côte-Sainte-Catherine", 45.4925, -73.6327),
    ("Plamondon", 45.4952, -73.6418), ("Namur", 45.4953, -73.6534), ("De la Savane", 45.5005, -73.6625),
    ("Du Collège", 45.5094, -73.6749), ("Côte-Vertu", 45.5143, -73.6821), ("Longueuil-U-de-S", 45.5245, -73.5215),
    ("Jean-Drapeau", 45.5134, -73.5332), ("Saint-Michel", 45.5597, -73.5997), ("D'Iberville", 45.5535, -73.6022),
    ("Fabre", 45.5469, -73.6074), ("De Castelnau", 45.5355, -73.6198), ("Parc", 45.5302, -73.6247),
    ("Acadie", 45.5235, -73.6234), ("Outremont", 45.5201, -73.6149), ("Édouard-Montpetit", 45.5100, -73.6128),
    ("Université-de-Montréal", 45.5030, -73.6181), ("Côte-des-Neiges", 45.4970, -73.6234), ("Cartier", 45.5604, -73.6817),
    ("De la Concorde", 45.5609, -73.7103), ("Montmorency", 45.5583, -73.7212)
]

metro_schema = StructType([
    StructField("metro_name", StringType(), True),
    StructField("m_lat", DoubleType(), True),
    StructField("m_lon", DoubleType(), True)
])
df_metro = spark.createDataFrame(metro_data, schema=metro_schema)

# 2. Cleanup: Remove existing columns to prevent AMBIGUOUS_REFERENCE
cols_to_clean = ["dist_km", "metro_name", "transit_proximity_tier"]
df_final = df_final.drop(*[c for c in cols_to_clean if c in df_final.columns])

# 3. Calculate Distances
print("📍 Calculating nearest Metro for each BIXI station...")
df_stations = df_final.select("station_id", "lat", "lon").distinct()
df_cross = df_stations.crossJoin(F.broadcast(df_metro))

def haversine_dist(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = F.radians(lat2 - lat1)
    dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat / 2)**2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2)**2
    return 2 * R * F.atan2(F.sqrt(a), F.sqrt(1 - a))

df_dist = df_cross.withColumn("dist_km_calc", haversine_dist(F.col("lat"), F.col("lon"), F.col("m_lat"), F.col("m_lon")))

# 4. Get Nearest Metro only
window_spec = Window.partitionBy("station_id").orderBy("dist_km_calc")
df_nearest = df_dist.withColumn("rn", F.row_number().over(window_spec)) \
    .filter("rn = 1") \
    .select("station_id", "metro_name", F.col("dist_km_calc").alias("dist_km"))

# 5. Join and Categorize Tiers
print("🚲 Mapping Transit Proximity Tiers...")
df_final = df_final.join(df_nearest, on="station_id", how="left")

df_final = df_final.withColumn("transit_proximity_tier", 
    F.when(F.col("dist_km") < 0.3, "Tier 1: High Proximity")
     .when(F.col("dist_km") <= 0.8, "Tier 2: Commuter Catchment")
     .otherwise("Tier 3: Transit Desert")
)

# 6. Verification
print("--- 📊 Final Tier Distribution ---")
df_final.select("station_id", "transit_proximity_tier").distinct() \
    .groupBy("transit_proximity_tier").count().orderBy("transit_proximity_tier").show()

# 7. Final Preview
df_final.select("station_id", "dist_km", "transit_proximity_tier").show(5)

📍 Calculating nearest Metro for each BIXI station...
🚲 Mapping Transit Proximity Tiers...
--- 📊 Final Tier Distribution ---


+----------------------+-----+
|transit_proximity_tier|count|
+----------------------+-----+
|  Tier 1: High Prox...|  443|
|  Tier 2: Commuter ...|  623|
|  Tier 3: Transit D...|  818|
+----------------------+-----+



+--------------------+-------------------+----------------------+
|          station_id|            dist_km|transit_proximity_tier|
+--------------------+-------------------+----------------------+
|45.524790,-73.565...| 0.7256533163973264|  Tier 2: Commuter ...|
|45.506539,-73.576...| 0.4817380565504807|  Tier 2: Commuter ...|
|45.470400,-73.565...|0.05858706141970477|  Tier 1: High Prox...|
|45.527831,-73.571...| 0.8704435413451855|  Tier 3: Transit D...|
|45.483249,-73.591...|  0.664270622598748|  Tier 2: Commuter ...|
+--------------------+-------------------+----------------------+
only showing top 5 rows


In [39]:
import pyspark.sql.functions as F

# 1. Preview the Column List and Types
print("--- 📑 Final Engineered Schema ---")
df_final.printSchema()

# 2. Preview a sample of the actual data
# We use vertical=True to make it easy to see all columns without scrolling horizontally
print("--- 🔍 Data Preview (First 2 Rows) ---")
df_final.show(2, truncate=False, vertical=True)

# 3. Quick Validation: Ensure no critical column is 100% Null
print("--- ✅ Feature Health Check ---")
df_final.select(
    (F.count("temp") / F.count("*") * 100).alias("weather_fill_%"),
    (F.count("transit_proximity_tier") / F.count("*") * 100).alias("transit_tier_fill_%"),
    (F.count("community_id") / F.count("*") * 100).alias("community_fill_%")
).show()

# 4. Save the "Golden" Dataset
# I am saving this to a 'final_features' folder so it's ready for your ML model
final_output_path = base_path / "data" / "gold" / "bixi_model_features_v1"
final_output_path.mkdir(parents=True, exist_ok=True)

print(f"💾 Saving final dataset to: {final_output_path}...")
df_final.write.mode("overwrite").parquet(f"file://{str(final_output_path)}")

print("🚀 Success! Your data is now ready for training.")

--- 📑 Final Engineered Schema ---
root
 |-- station_id: string (nullable = true)
 |-- ts_hour: timestamp (nullable = true)
 |-- demand_count: long (nullable = false)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- is_holiday: integer (nullable = false)
 |-- community_id: long (nullable = true)
 |-- temp: float (nullable = true)
 |-- precip: float (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- metro_proximity: string (nullable = false)
 |-- metro_name: string (nullable = true)
 |-- dist_km: double (nullable = true)
 |-- transit_proximity_tier: string (nullable = false)

--- 🔍 Data Preview (First 2 Rows) ---


-RECORD 0-------------------------------------------------------------------
 station_id             | 45.524790,-73.565450|calixa-lavallée / sherbrooke 
 ts_hour                | 2024-06-24 16:00:00                               
 demand_count           | 9                                                 
 day_of_week            | 2                                                 
 is_weekend             | 0                                                 
 is_holiday             | 1                                                 
 community_id           | NULL                                              
 temp                   | 21.4                                              
 precip                 | 0.0                                               
 lat                    | 45.52479                                          
 lon                    | -73.56545                                         
 metro_proximity        | High                                              

+-----------------+-------------------+----------------+
|   weather_fill_%|transit_tier_fill_%|community_fill_%|
+-----------------+-------------------+----------------+
|99.37975100390558|              100.0|             0.0|
+-----------------+-------------------+----------------+

💾 Saving final dataset to: /Users/comanetan/projects/bixi-analytics/data/gold/bixi_model_features_v1...


🚀 Success! Your data is now ready for training.
